# Treinamento dos Modelos com MLflow

Este notebook tem como objetivo treinar diferentes arquiteturas de redes neurais para classificação de doenças em folhas de milho.

Nesta etapa serão executados:

1. Configuração do ambiente;
2. Carregamento dos DataLoaders;
3. Teste de diferentes arquiteturas;
4. Busca de learning rate com LR Finder;
5. Treinamento com learning rate fixo;
6. Treinamento com One Cycle Policy;
7. Registro dos experimentos no MLflow;
8. Avaliação por matriz de confusão;
9. Comparação dos modelos treinados.

Cada experimento será registrado no MLflow para permitir rastreabilidade e comparação entre modelos.

Serão testados:

- **SimpleCNN:**

    Modelo criado do zero para o problema em questão.

- **ResNet-18:**

    Rede neural convolucional profunda pré treinada com 18 camadas.

- **EfficientNet-b0:**

    Faz parte da família EfficientNet desenvolvida pelo Google, sendo b0 um modelo leve e de alta eficiência.

- **DenseNet-121:**

    Rede neural convolucional profunda, famosa pela arquitetura densamente conectada. Opera melhor com bases de dados maiores, será testada apenas como experimento.

Para efeitos de comparação, em cada run serão logados os seguintes parametros:

- Nome do modelo;
- Taxa de aprendizado;
- Freeze Backbone;
- Épocas;
- Classes;
- Early stopping (caso ocorra)

Para avaliação do desempenho serão logadas as métricas train_loss, train_acc, val_loss, val_acc.

Para cada modelo, serão treinadas duas versões distintas. A primeira utilizará o learning rate definido a partir do lr_finder, enquanto a segunda será treinada aplicando a estratégia One Cycle Policy. O objetivo é comparar o desempenho entre as duas abordagens e selecionar a configuração que apresentar os melhores resultados.

## 1. Imports e configuração inicial

Nesta etapa importamos as bibliotecas principais utilizadas no treinamento, avaliação e registro dos experimentos.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

from PIL import Image
import io
import tempfile
import os

import torch
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from src.train import find_best_lr, run_training
from src.dataset.milho_dataset import get_dataloaders_milho
from src.evaluate import get_predictions
from src.models.milho_models import get_model

## 2. Modelo criado do zero (Baseline)

Nesta seção utilizamos uma arquitetura de rede neural convolucional desenvolvida do zero (`SimpleCNN`).

O objetivo deste modelo é servir como **baseline**, ou seja:

- estabelecer um ponto de referência de desempenho;
- permitir comparação com modelos pré-treinados;
- entender a capacidade do modelo em aprender padrões diretamente dos dados, sem transfer learning.

### Learning Rate Finder — CNN Simples

Nesta etapa utilizamos o **Learning Rate Finder (LR Finder)** para identificar uma taxa de aprendizado adequada para o treinamento da `SimpleCNN`.

O LR Finder funciona testando múltiplos valores de learning rate ao longo de um único ciclo de treinamento e observando o comportamento da função de perda.

#### Objetivo desta etapa:

- Encontrar um intervalo de learning rates estáveis;
- Evitar valores muito baixos (treinamento lento);
- Evitar valores muito altos (divergência do modelo);
- Definir um valor inicial mais eficiente para o treinamento.

#### Configuração do experimento:

- Modelo: `SimpleCNN`
- Estratégia: Treinamento do zero (sem transfer learning)
- Backbone congelado: Não aplicável, mas mantido como padrão
- Dataset: milho (dados já pré-processados e divididos)
- Registro: MLflow

O resultado será um gráfico que relaciona:

- Learning Rate vs Loss

A partir desse gráfico, será possível escolher um learning rate adequado para os próximos experimentos.

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "SimpleCNN"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

In [0]:
run_id = "085eca5d71904b6dbf0c9339df7b3d9f"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr = float(params.get("suggested_lr"))
suggested_lr

O melhor Leanring Rate encotrado para o modelo foi de 0.0014849682622544637.

### Treinamento da CNN com Learning Rate fixo:

Nesta etapa será realizado o treinamento da arquitetura `SimpleCNN` utilizando um **learning rate fixo**, previamente determinado através do LR Finder.

Este experimento tem como objetivo:

- Avaliar o desempenho do modelo utilizando uma taxa de aprendizado constante;
- Servir como baseline para comparação com estratégias mais avançadas (ex: One Cycle Policy);
- Observar a estabilidade do treinamento ao longo das épocas.

O modelo será treinado utilizando:

- Dados de treino e validação já pré-processados;
- Early stopping baseado em `patience`;
- Monitoramento de métricas a cada época;
- Registro completo no MLflow.

### Configuração do experimento

- **Modelo:** `SimpleCNN`
- **Learning Rate:** valor sugerido pelo LR Finder
- **Épocas:** 30
- **Early Stopping:** ativado (`patience = 5`)
- **Estratégia de LR:** fixa 
- **Backbone congelado:** não aplicável (modelo customizado), mas mantido por padronização
- **Dataset:** milho

### Durante o treinamento serão registrados:

- Loss de treino (`train_loss`)
- Acurácia de treino (`train_acc`)
- Loss de validação (`val_loss`)
- Acurácia de validação (`val_acc`)

### Pós-treinamento

Após o treinamento:

- O modelo será salvo no MLflow;
- Será registrada a assinatura de entrada e saída (`signature`);
- Um exemplo de input será armazenado para facilitar futuras inferências.

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "SimpleCNN"
learning_rate = suggested_lr
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_CNN, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_CNN(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_CNN,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

Durante o treinamento o Early stopping foi acionado, já que não houve melhora na acurácia da validação.
Das 30 épocas programadas para o treinamento, foram concluídas 21.

### Matriz de Confusão — CNN (Learning Rate fixo)

Após o treinamento do modelo, é fundamental avaliar não apenas a acurácia geral, mas também como o modelo está se comportando em cada classe individualmente.

Para isso, utilizamos a **matriz de confusão** no conjunto de validação.

In [0]:
run_id_CNN = "6d60be5f39024202b9fb0863811c8632"

with mlflow.start_run(run_id=run_id_CNN):

    val_labels, val_preds = get_predictions(model_CNN, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_Simple_CNN.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_Simple_CNN.png")

### CNN com One Cycle Policy

Nesta etapa será realizado o treinamento da arquitetura `SimpleCNN` utilizando a estratégia de **One Cycle Policy**.

Diferente do treinamento com learning rate fixo, aqui o learning rate **varia dinamicamente ao longo das épocas**, seguindo um ciclo pré-definido.

A **One Cycle Policy** é uma estratégia de ajuste de learning rate que consiste em:

- Aumentar gradualmente o learning rate até um valor máximo;
- Reduzir o learning rate progressivamente até valores baixos ao final do treinamento.

#### Objetivos desta abordagem:

- Melhorar a convergência do modelo;
- Evitar mínimos locais;
- Acelerar o treinamento;
- Melhorar a capacidade de generalização.

#### Fluxo da estratégia:

1. Fase de aquecimento (learning rate crescente);
2. Pico de aprendizado;
3. Decaimento gradual do learning rate.

---

#### Configuração do experimento:

- **Modelo:** `SimpleCNN`
- **Learning Rate inicial:** obtido via LR Finder
- **Estratégia:** One Cycle Policy
- **Épocas:** 25
- **Dataset:** milho
- **Registro:** MLflow

#### Integração com MLflow

O learning rate utilizado é recuperado automaticamente a partir de uma execução anterior do **LR Finder**, garantindo consistência entre os experimentos.

Durante o treinamento serão registrados:

- Loss de treino (`train_loss`)
- Acurácia de treino (`train_acc`)
- Loss de validação (`val_loss`)
- Acurácia de validação (`val_acc`)

Além disso:

- O modelo treinado será salvo;
- A assinatura de entrada e saída será registrada;
- Um exemplo de input será armazenado para futuras inferências.

---

#### Objetivo desta comparação

Este experimento será comparado com o treinamento anterior (learning rate fixo) para avaliar:

- estabilidade do treinamento;
- velocidade de convergência;
- desempenho em validação;
- comportamento das métricas ao longo das épocas.

Essa comparação permitirá entender se o uso do One Cycle Policy traz ganhos reais para este problema.

In [0]:
run_id = "085eca5d71904b6dbf0c9339df7b3d9f"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr = float(params.get("suggested_lr"))

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "SimpleCNN"
learning_rate = suggested_lr
epochs = 25
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_CNN_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_CNN_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_CNN_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

### Matriz de Confusão — CNN (One Cycle Policy)

Após o treinamento da `SimpleCNN` utilizando a estratégia de **One Cycle Policy**, realizamos a avaliação do modelo no conjunto de validação por meio da matriz de confusão.

Esta etapa complementa a análise das métricas agregadas, permitindo uma visão mais detalhada do comportamento do modelo em cada classe.

#### Registro no MLflow

A matriz de confusão gerada será:

- salva como imagem;
- registrada como artefato na execução correspondente no MLflow;

permitindo consulta futura e comparação entre experimentos.

In [0]:
run_id_CNN_1cycle = "39efe20dccaf4baf9c33e4e128c93067"

with mlflow.start_run(run_id=run_id_CNN_1cycle):

    val_labels, val_preds = get_predictions(model_CNN_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_Simple_CNN_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_Simple_CNN_1cycle.png")

## 3. Modelo pré-treinado (Transfer Learning) — ResNet18

Nesta seção utilizamos a arquitetura `ResNet18`, um modelo pré-treinado no dataset ImageNet, amplamente utilizado em tarefas de visão computacional.

O objetivo deste modelo é aplicar **transfer learning**, ou seja:

- aproveitar padrões já aprendidos em grandes volumes de dados;
- reduzir o tempo de treinamento;
- melhorar a capacidade de generalização;
- obter melhor desempenho em comparação com modelos treinados do zero.

Neste experimento, utilizamos a estratégia de:

- **Backbone congelado (`freeze_backbone = True`)**

Isso significa que:

- as camadas convolucionais permanecem fixas;
- apenas a camada final de classificação é treinada;
- o modelo adapta o conhecimento prévio ao problema específico (classificação de doenças no milho).

Este modelo será comparado com a CNN simples para avaliar os ganhos obtidos com o uso de transfer learning.

### Learning Rate Finder — ResNet18

Nesta etapa utilizamos o **Learning Rate Finder (LR Finder)** para identificar uma taxa de aprendizado adequada para o treinamento da `ResNet18`.

Diferente da CNN simples, a `ResNet18` é um modelo **pré-treinado no ImageNet**, permitindo a aplicação de **transfer learning**, onde aproveitamos padrões já aprendidos em um grande volume de dados.

#### Objetivo desta etapa:

- Encontrar um intervalo de learning rates estáveis para fine-tuning;
- Evitar valores muito baixos (treinamento lento);
- Evitar valores muito altos (instabilidade ou destruição dos pesos pré-treinados);
- Definir um learning rate inicial adequado para o treinamento da camada final.

#### Configuração do experimento:

- Modelo: `ResNet18`
- Estratégia: Transfer learning (backbone congelado)
- Backbone congelado: Sim (`freeze_backbone = True`)
- Dataset: milho
- Registro: MLflow

O resultado será um gráfico que relaciona:

- Learning Rate vs Loss

A partir desse gráfico, será possível escolher um learning rate adequado para os próximos experimentos com a ResNet18.

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "resnet18"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

### Treinamento — ResNet18 com Learning Rate fixo

Nesta etapa será realizado o treinamento da `ResNet18` utilizando um **learning rate fixo**, definido a partir do valor sugerido pelo LR Finder.

Como estamos utilizando um modelo pré-treinado, o learning rate sugerido foi reduzido (`suggested_lr / 10`) para tornar o ajuste mais conservador e evitar grandes alterações nos pesos aprendidos previamente.

#### Objetivo desta etapa:

- Treinar a camada final de classificação da `ResNet18`;
- Avaliar o desempenho do modelo com transfer learning;
- Comparar os resultados com a CNN simples treinada do zero;
- Verificar se o uso de pesos pré-treinados melhora a generalização.

#### Configuração do experimento:

- Modelo: `ResNet18`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Learning Rate: valor sugerido pelo LR Finder dividido por 10
- Épocas: 30
- Early stopping: Sim (`patience = 5`)
- One Cycle Policy: Não
- Dataset: milho
- Registro: MLflow

Durante o treinamento serão registradas no MLflow as métricas de:

- Loss de treino;
- Acurácia de treino;
- Loss de validação;
- Acurácia de validação.

Ao final, o modelo treinado será salvo no MLflow junto com sua assinatura de entrada e saída.

In [0]:
run_id = "97cb6892487643f28c4c42bcf3b198be"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_resnet = float(params.get("suggested_lr")) / 10
suggested_lr_resnet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "resnet18"
learning_rate = suggested_lr_resnet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_resnet18, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_resnet18(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_resnet18,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

### Matriz de Confusão — ResNet18 (Learning Rate fixo)

Após o treinamento da `ResNet18` com learning rate fixo, realizamos a avaliação do modelo no conjunto de validação utilizando a matriz de confusão.

Essa análise permite ir além das métricas agregadas, como acurácia, e entender como o modelo está se comportando em cada classe individualmente.

In [0]:
run_id_resnet18 = "ecfe31e2a87c42cab69b824c7e7b4fa2"

with mlflow.start_run(run_id=run_id_resnet18):

    val_labels, val_preds = get_predictions(model_resnet18, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_resnet18.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_resnet18.png")


### ResNet18 com One Cycle Policy

Nesta etapa será realizado o treinamento da `ResNet18` utilizando a estratégia de **One Cycle Policy**.

Diferente do treinamento com learning rate fixo, essa abordagem ajusta dinamicamente a taxa de aprendizado ao longo das épocas, alternando entre fases de aumento e redução do learning rate.

#### Objetivo desta etapa:

- Avaliar o desempenho da `ResNet18` com uma estratégia dinâmica de learning rate;
- Verificar se a One Cycle Policy melhora a convergência do modelo;
- Comparar os resultados com o treinamento da ResNet18 usando learning rate fixo;
- Analisar se a estratégia contribui para melhor generalização no conjunto de validação.

#### Configuração do experimento:

- Modelo: `ResNet18`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Learning Rate: valor ajustado a partir do LR Finder
- Épocas: 30
- One Cycle Policy: Sim
- Dataset: milho (dados já pré-processados e divididos)
- Registro: MLflow

Durante o treinamento serão registradas no MLflow as métricas de:

- Loss de treino;
- Acurácia de treino;
- Loss de validação;
- Acurácia de validação.

Ao final, o modelo treinado será salvo no MLflow junto com sua assinatura de entrada e saída, permitindo rastreabilidade e comparação com os demais experimentos.

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "resnet18"
learning_rate = suggested_lr_resnet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_resnet18_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_resnet18_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_resnet18_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

### Matriz de Confusão — ResNet18 (One Cycle Policy)

Após o treinamento da `ResNet18` utilizando a estratégia de **One Cycle Policy**, realizamos a avaliação do modelo no conjunto de validação por meio da matriz de confusão.

Essa análise permite observar o comportamento do modelo em cada classe individualmente, complementando métricas agregadas como acurácia e loss.

In [0]:
run_id_resnet18_1cycle = "67254b3ec1ec4a0b97a5835eafaee785"
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
model_resnet18_1cycle = mlflow.pytorch.load_model(f"runs:/{run_id_resnet18_1cycle}/model")
_, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with mlflow.start_run(run_id=run_id_resnet18_1cycle):

    val_labels, val_preds = get_predictions(model_resnet18_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_resnet18_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_resnet18_1cycle.png")

## 4. Modelo pré-treinado (Transfer Learning) — EfficientNet-B0

Nesta seção utilizamos a arquitetura `EfficientNet-B0`, um modelo pré-treinado no dataset ImageNet e desenvolvido com foco em eficiência computacional e alto desempenho em tarefas de visão computacional.

O objetivo deste modelo é aplicar **transfer learning**, ou seja:

- aproveitar padrões previamente aprendidos em grandes datasets;
- reduzir o tempo de treinamento;
- melhorar a capacidade de generalização;
- obter melhor desempenho utilizando uma arquitetura mais eficiente.

A `EfficientNet-B0` utiliza uma estratégia de escalonamento balanceado entre:

- profundidade da rede;
- largura das camadas;
- resolução das imagens.

Isso permite alcançar bons resultados com menor custo computacional em comparação com arquiteturas mais pesadas.

Neste experimento, utilizamos a estratégia de:

- **Backbone congelado (`freeze_backbone = True`)**

Isso significa que:

- as camadas convolucionais pré-treinadas permanecem fixas;
- apenas a camada final de classificação será treinada;
- o modelo adapta o conhecimento aprendido no ImageNet ao problema específico de classificação de doenças no milho.

Este modelo será comparado com:

- CNN simples (baseline);
- ResNet18;

Permitindo avaliar os ganhos obtidos com arquiteturas mais eficientes e o uso de transfer learning.

### Learning Rate Finder — EfficientNet-B0

Utilizamos o **Learning Rate Finder (LR Finder)** para identificar uma taxa de aprendizado adequada para o treinamento da `EfficientNet-B0`.

O LR Finder funciona testando múltiplos valores de learning rate ao longo de um único ciclo de treinamento e observando o comportamento da função de perda.

#### Objetivo desta etapa:

- Encontrar um intervalo de learning rates estáveis para fine-tuning;
- Evitar valores muito baixos (treinamento lento);
- Evitar valores muito altos (instabilidade ou degradação dos pesos pré-treinados);
- Definir um learning rate inicial adequado para o treinamento da camada final.

#### Configuração do experimento:

- Modelo: `EfficientNet-B0`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Dataset: milho (dados já pré-processados e divididos)
- Registro: MLflow

O resultado será um gráfico que relaciona:

- Learning Rate vs Loss

A partir desse gráfico, será possível escolher um learning rate adequado para os próximos experimentos com a EfficientNet-B0.

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "efficientnet_b0"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

### Treinamento — EfficientNet-B0 com Learning Rate fixo

Nesta etapa será realizado o treinamento da `EfficientNet-B0` utilizando um **learning rate fixo**, definido a partir do valor sugerido pelo LR Finder.

Como a `EfficientNet-B0` é um modelo pré-treinado no ImageNet, utilizamos **transfer learning** com o backbone congelado. Dessa forma, apenas a camada final de classificação será ajustada para o problema específico de classificação de doenças no milho.

#### Objetivo desta etapa:

- Treinar a camada final da `EfficientNet-B0`;
- Avaliar o desempenho de uma arquitetura eficiente com transfer learning;
- Comparar os resultados com a CNN simples e a ResNet18;
- Verificar se a EfficientNet-B0 apresenta melhor equilíbrio entre desempenho e custo computacional.

#### Configuração do experimento:

- Modelo: `EfficientNet-B0`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Learning Rate: valor sugerido pelo LR Finder
- Épocas: 30
- Early stopping: Sim
- One Cycle Policy: Não
- Dataset: milho
- Registro: MLflow

Durante o treinamento serão registradas no MLflow as métricas de:

- Loss de treino;
- Acurácia de treino;
- Loss de validação;
- Acurácia de validação.

Ao final, o modelo treinado será salvo no MLflow junto com sua assinatura de entrada e saída, permitindo rastreabilidade e comparação com os demais experimentos.

In [0]:
run_id = "9d6edde8c6694ea5a888ab70dcf04695"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_effnet = float(params.get("suggested_lr"))
suggested_lr_effnet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "efficientnet_b0"
learning_rate = suggested_lr_effnet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_effnetb0, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_effnetb0(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_effnetb0,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

### Matriz de Confusão — EfficientNet-B0 (Learning Rate fixo)

Após o treinamento da `EfficientNet-B0` com learning rate fixo, realizamos a avaliação do modelo no conjunto de validação utilizando a matriz de confusão.

Essa análise permite observar de forma mais detalhada como o modelo está classificando cada classe individualmente, complementando métricas agregadas como acurácia e loss.

In [0]:
run_id_effnet = "a6af598f96c243e9b60e29937eb6aec5"

with mlflow.start_run(run_id=run_id_effnet):

    val_labels, val_preds = get_predictions(model_effnetb0, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_efnet.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_efnet.png")

### EfficientNet-B0 com One Cycle Policy

Nesta etapa será realizado o treinamento da `EfficientNet-B0` utilizando a estratégia de **One Cycle Policy**.

Diferente do treinamento com learning rate fixo, essa abordagem ajusta dinamicamente a taxa de aprendizado ao longo das épocas, buscando melhorar a convergência e a capacidade de generalização do modelo.

#### Objetivo desta etapa:

- Avaliar o desempenho da `EfficientNet-B0` com uma estratégia dinâmica de learning rate;
- Verificar se a One Cycle Policy melhora a estabilidade do treinamento;
- Comparar os resultados com a EfficientNet-B0 treinada com learning rate fixo;
- Analisar se a combinação entre transfer learning e One Cycle Policy melhora a performance no conjunto de validação.

#### Configuração do experimento:

- Modelo: `EfficientNet-B0`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Learning Rate: valor sugerido pelo LR Finder
- Épocas: 30
- One Cycle Policy: Sim
- Dataset: milho
- Registro: MLflow

Durante o treinamento serão registradas no MLflow as métricas de:

- Loss de treino;
- Acurácia de treino;
- Loss de validação;
- Acurácia de validação.

Ao final, o modelo treinado será salvo no MLflow junto com sua assinatura de entrada e saída, permitindo rastreabilidade e comparação com os demais experimentos.

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "efficientnet_b0"
learning_rate = suggested_lr_effnet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_effnetb0_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_effnetb0_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_effnetb0_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

### Matriz de Confusão — EfficientNet-B0 (One Cycle Policy)

Após o treinamento da `EfficientNet-B0` utilizando a estratégia de **One Cycle Policy**, realizamos a avaliação do modelo no conjunto de validação por meio da matriz de confusão.

Essa análise permite observar com mais detalhes como o modelo está classificando cada classe individualmente, complementando métricas agregadas como acurácia e loss.

In [0]:
run_id_effnet_1cycle = "e3f855c9716c47dab3f623a344e6ae05"

with mlflow.start_run(run_id=run_id_effnet_1cycle):

    val_labels, val_preds = get_predictions(model_effnetb0_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_efnet_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_efnet_1cycle.png")

## 5. Modelo pré-treinado (Transfer Learning) — DenseNet121

Nesta seção utilizamos a arquitetura `DenseNet121`, um modelo pré-treinado no dataset ImageNet e amplamente utilizado em tarefas de visão computacional devido à sua capacidade de reutilização de características ao longo da rede.

O objetivo deste modelo é aplicar **transfer learning**, ou seja:

- aproveitar padrões previamente aprendidos em grandes datasets;
- reduzir o tempo de treinamento;
- melhorar a capacidade de generalização;
- explorar uma arquitetura mais profunda e densamente conectada.

A `DenseNet121` utiliza conexões densas entre as camadas da rede, permitindo que:

- características aprendidas sejam reutilizadas ao longo do modelo;
- o fluxo de gradientes seja mais eficiente;
- a perda de informação entre camadas seja reduzida.

Essa arquitetura tende a apresentar bom desempenho em problemas de classificação de imagens, especialmente em cenários com padrões visuais complexos.

Neste experimento, utilizamos a estratégia de:

- **Backbone congelado (`freeze_backbone = True`)**

Isso significa que:

- as camadas convolucionais pré-treinadas permanecem fixas;
- apenas a camada final de classificação será treinada;
- o modelo adapta o conhecimento aprendido no ImageNet ao problema específico de classificação de doenças no milho.

Este modelo será comparado com:

- CNN simples (baseline);
- ResNet18;
- EfficientNet-B0;

Permitindo avaliar os ganhos obtidos com arquiteturas mais profundas e o uso de transfer learning.

### Learning Rate Finder — DenseNet121

Utilizamos o **Learning Rate Finder (LR Finder)** para identificar uma taxa de aprendizado adequada para o treinamento da `DenseNet121`.

O LR Finder funciona testando múltiplos valores de learning rate ao longo de um único ciclo de treinamento e observando o comportamento da função de perda.


#### Objetivo desta etapa:

- Encontrar um intervalo de learning rates estáveis para fine-tuning;
- Evitar valores muito baixos (treinamento lento);
- Evitar valores muito altos (instabilidade ou degradação dos pesos pré-treinados);
- Definir um learning rate inicial adequado para o treinamento da camada final.

#### Configuração do experimento:

- Modelo: `DenseNet121`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Dataset: milho
- Registro: MLflow

O resultado será um gráfico que relaciona:

- Learning Rate vs Loss

A partir desse gráfico, será possível escolher um learning rate adequado para os próximos experimentos com a DenseNet121.

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "densenet121"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

### Treinamento — DenseNet121 com Learning Rate fixo

Nesta etapa será realizado o treinamento da `DenseNet121` utilizando um **learning rate fixo**, definido a partir do valor sugerido pelo LR Finder.

#### Objetivo desta etapa:

- Treinar a camada final da `DenseNet121`;
- Avaliar o desempenho de uma arquitetura mais profunda com transfer learning;
- Comparar os resultados com a CNN simples, ResNet18 e EfficientNet-B0;
- Verificar se as conexões densas da arquitetura contribuem para melhor generalização.

#### Configuração do experimento:

- Modelo: `DenseNet121`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Learning Rate: valor sugerido pelo LR Finder
- Épocas: 30
- Early stopping: Sim
- One Cycle Policy: Não
- Dataset: milho
- Registro: MLflow

Durante o treinamento serão registradas no MLflow as métricas de:

- Loss de treino;
- Acurácia de treino;
- Loss de validação;
- Acurácia de validação.

Ao final, o modelo treinado será salvo no MLflow junto com sua assinatura de entrada e saída, permitindo rastreabilidade e comparação com os demais experimentos.

In [0]:
run_id = "9375f9c7b8284e458de6969fabda6a0e"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_densenet = float(params.get("suggested_lr"))
suggested_lr_densenet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "densenet121"
learning_rate = suggested_lr_densenet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_densenet, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_densenet(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_densenet,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

### Matriz de Confusão — DenseNet121 (Learning Rate fixo)

Após o treinamento da `DenseNet121` com learning rate fixo, realizamos a avaliação do modelo no conjunto de validação utilizando a matriz de confusão.

Essa análise permite observar com maior profundidade como o modelo está classificando cada classe individualmente, complementando métricas agregadas como acurácia e loss.

In [0]:
run_id_densenet = "aafaa31d71f5460d8cb78214cd97f94d"

with mlflow.start_run(run_id=run_id_densenet):

    val_labels, val_preds = get_predictions(model_densenet, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_dense.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_dense.png")

### DenseNet121 com One Cycle Policy

Nesta etapa será realizado o treinamento da `DenseNet121` utilizando a estratégia de **One Cycle Policy**.

Diferente do treinamento com learning rate fixo, essa abordagem ajusta dinamicamente a taxa de aprendizado ao longo das épocas, buscando melhorar a convergência e a capacidade de generalização do modelo.

#### Objetivo desta etapa:

- Avaliar o desempenho da `DenseNet121` com uma estratégia dinâmica de learning rate;
- Verificar se a One Cycle Policy melhora a estabilidade do treinamento;
- Comparar os resultados com a DenseNet121 treinada com learning rate fixo;
- Analisar se a combinação entre transfer learning, conexões densas e One Cycle Policy melhora a performance no conjunto de validação.

#### Configuração do experimento:

- Modelo: `DenseNet121`
- Estratégia: Transfer learning
- Backbone congelado: Sim (`freeze_backbone = True`)
- Learning Rate: valor sugerido pelo LR Finder
- Épocas: 30
- One Cycle Policy: Sim
- Dataset: milho
- Registro: MLflow

Durante o treinamento serão registradas no MLflow as métricas de:

- Loss de treino;
- Acurácia de treino;
- Loss de validação;
- Acurácia de validação.

Ao final, o modelo treinado será salvo no MLflow junto com sua assinatura de entrada e saída, permitindo rastreabilidade e comparação com os demais experimentos.

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "densenet121"
learning_rate = suggested_lr_densenet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_densenet_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_densenet_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_densenet_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

### Matriz de Confusão — DenseNet121 (One Cycle Policy)

Após o treinamento da `DenseNet121` utilizando a estratégia de **One Cycle Policy**, realizamos a avaliação do modelo no conjunto de validação por meio da matriz de confusão.

Nesta etapa, o modelo é carregado diretamente a partir da execução registrada no MLflow, garantindo que a avaliação seja feita sobre o artefato salvo durante o experimento.


In [0]:
run_id_densenet_1cycle = "5af183c6045c4698816dd0f6d711bc4e"
artifact_path = "model"
model_uri = f"runs:/{run_id_densenet_1cycle}/{artifact_path}"
model_densenet_1cycle = mlflow.pytorch.load_model(model_uri)

data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
_ , val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

with mlflow.start_run(run_id=run_id_densenet_1cycle):


    val_labels, val_preds = get_predictions(model_densenet_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_dense_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_dense_1cycle.png")

## 6. Análise comparativa dos experimentos

Após a execução dos diferentes treinamentos, torna-se necessário realizar uma análise consolidada dos resultados obtidos ao longo dos experimentos.

Nesta etapa serão comparados os modelos treinados utilizando:

- arquiteturas distintas;
- estratégias de learning rate;
- diferentes abordagens de treinamento.

A análise será baseada principalmente nas métricas obtidas no conjunto de validação, permitindo identificar quais modelos apresentaram melhor capacidade de generalização.

---

#### Objetivos desta etapa

- Comparar o desempenho entre os modelos treinados;
- Avaliar o impacto do transfer learning;
- Verificar os efeitos da One Cycle Policy;
- Identificar possíveis sinais de overfitting;
- Selecionar os experimentos com melhor equilíbrio entre acurácia e loss.

---

#### Critérios avaliados

Os experimentos serão analisados considerando:

- acurácia de treino;
- acurácia de validação;
- loss de treino;
- loss de validação;
- estabilidade do treinamento;
- capacidade de generalização.

Além disso, será utilizado um score auxiliar para facilitar o ranqueamento dos modelos com melhor desempenho geral.

---

#### Objetivo final

Ao final desta análise será possível identificar:

- quais arquiteturas apresentaram melhor desempenho;
- quais estratégias de treinamento foram mais eficientes;
- quais modelos possuem maior potencial para uso em inferência e deploy.

In [0]:
experiment_name = "/Shared/Milho_Doencas"

experiment = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_compare = runs[[
    "run_id",
    "params.model_name",
    "params.learning_rate",
    "params.use_onecycle",
    "params.epochs",
    "metrics.train_acc",
    "metrics.val_acc",
    "metrics.train_loss",
    "metrics.val_loss"
]].copy()

df_compare.columns = [
    "run_id",
    "model",
    "lr",
    "onecycle",
    "epochs",
    "train_acc",
    "val_acc",
    "train_loss",
    "val_loss"
]

df_compare["score"] = (
    df_compare["val_acc"] - 0.1 * df_compare["val_loss"]
)

df_compare = df_compare.sort_values(
    by="score",
    ascending=False
)

df_compare.head(10)

,run_id,model,lr,onecycle,epochs,train_acc,val_acc,train_loss,val_loss,score
10,39efe20dccaf4baf9c33e4e128c93067,SimpleCNN,0.0014849682622544637,True,25,0.892418,0.926752,0.273464,0.200815,0.906670
1,5af183c6045c4698816dd0f6d711bc4e,densenet121,0.005462277217684344,True,30,0.903689,0.917197,0.244502,0.261865,0.891011
4,e3f855c9716c47dab3f623a344e6ae05,efficientnet_b0,0.012328467394420665,True,30,0.879440,0.912420,0.338133,0.287337,0.883687
11,6d60be5f39024202b9fb0863811c8632,SimpleCNN,0.0014849682622544637,False,30,0.861680,0.896497,0.345454,0.249974,0.871499
7,67254b3ec1ec4a0b97a5835eafaee785,resnet18,0.027185882427329423,True,30,0.886954,0.904459,0.332407,0.354344,0.869024
5,a6af598f96c243e9b60e29937eb6aec5,efficientnet_b0,0.012328467394420665,False,30,0.843579,0.899682,0.542053,0.346011,0.865080
2,aafaa31d71f5460d8cb78214cd97f94d,densenet121,0.005462277217684344,False,30,0.870902,0.894904,0.319004,0.351505,0.859754
8,ecfe31e2a87c42cab69b824c7e7b4fa2,resnet18,0.027185882427329423,False,30,0.842896,0.875796,0.715594,0.694460,0.806350
0,967be9fed37b4feaa1368c47368e363e,None,None,None,None,NaN,NaN,NaN,NaN,NaN
3,9375f9c7b8284e458de6969fabda6a0e,densenet121,None,None,None,NaN,NaN,NaN,NaN,NaN


### Seleção do Modelo

A escolha do modelo final foi realizada com base em uma análise quantitativa e qualitativa dos experimentos registrados no MLflow.

#### Estratégia de seleção:

Os modelos foram ranqueados utilizando um score combinado:

score = val_acc - 0.1 * val_loss

Essa abordagem permite:

- Priorizar modelos com alta acurácia
- Penalizar modelos com baixa confiança (loss elevada)

#### Critério final:

1. Maior score combinado  
2. Menor loss de validação (em caso de empate)  
3. Análise da matriz de confusão  

In [0]:
best_run = df_compare.iloc[0]

print("🏆 Melhor modelo encontrado:")
print(best_run)

🏆 Melhor modelo encontrado:
run_id        39efe20dccaf4baf9c33e4e128c93067
model                                SimpleCNN
lr                       0.0014849682622544637
onecycle                                  True
epochs                                      25
train_acc                             0.892418
val_acc                               0.926752
train_loss                            0.273464
val_loss                              0.200815
score                                  0.90667
Name: 10, dtype: object
